# NLP Weekly Task 4: Bag of Words & TF-IDF

Implement the concept of **Bag of Words (BoW)** and **TF-IDF** using Python libraries (`scikit-learn`).

**Requirements:**
```
pip install scikit-learn pandas
```


In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

## 1. Sample Corpus

We'll use a small set of sentences as our corpus.

In [2]:
corpus = [
    "The cat sat on the mat",
    "The dog sat on the log",
    "Cats and dogs are great pets",
    "I love my pet cat and my pet dog"
]

for i, doc in enumerate(corpus):
    print(f"Doc {i}: {doc}")

Doc 0: The cat sat on the mat
Doc 1: The dog sat on the log
Doc 2: Cats and dogs are great pets
Doc 3: I love my pet cat and my pet dog


## 2. Bag of Words (BoW)

BoW represents each document as a vector of **raw word counts**, ignoring grammar and word order — only frequency matters.

In [3]:
bow_vectorizer = CountVectorizer(lowercase=True, stop_words=None)
bow_matrix = bow_vectorizer.fit_transform(corpus)

bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=bow_vectorizer.get_feature_names_out(),
    index=[f"Doc {i}" for i in range(len(corpus))]
)

print("Vocabulary:")
print(bow_vectorizer.get_feature_names_out())
bow_df

Vocabulary:
['and' 'are' 'cat' 'cats' 'dog' 'dogs' 'great' 'log' 'love' 'mat' 'my'
 'on' 'pet' 'pets' 'sat' 'the']


,and,are,cat,cats,dog,dogs,great,log,love,mat,my,on,pet,pets,sat,the
Doc 0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,1,2
Doc 1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,1,2
Doc 2,1,1,0,1,0,1,1,0,0,0,0,0,0,1,0,0
Doc 3,1,0,1,0,1,0,0,0,1,0,2,0,2,0,0,0


### BoW with stop words removed

Removing common English stop words (the, on, and, ...) gives a cleaner, more informative vector.

In [4]:
bow_vectorizer_sw = CountVectorizer(lowercase=True, stop_words="english")
bow_matrix_sw = bow_vectorizer_sw.fit_transform(corpus)

bow_df_sw = pd.DataFrame(
    bow_matrix_sw.toarray(),
    columns=bow_vectorizer_sw.get_feature_names_out(),
    index=[f"Doc {i}" for i in range(len(corpus))]
)
bow_df_sw

,cat,cats,dog,dogs,great,log,love,mat,pet,pets,sat
Doc 0,1,0,0,0,0,0,0,1,0,0,1
Doc 1,0,0,1,0,0,1,0,0,0,0,1
Doc 2,0,1,0,1,1,0,0,0,0,1,0
Doc 3,1,0,1,0,0,0,1,0,2,0,0


## 3. TF-IDF

TF-IDF weighs each word by how important it is to a document relative to the whole corpus:

- **TF** = (word count in doc) / (total words in doc)
- **IDF** = log((1 + N) / (1 + docs containing word)) + 1  *(sklearn's smoothed formula)*
- **TF-IDF** = TF × IDF

Words common across **all** documents (like "the", "sat", "on") get a **lower** score, while words distinctive to a document get a **higher** score.

In [5]:
tfidf_vectorizer = TfidfVectorizer(lowercase=True, stop_words=None)
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=[f"Doc {i}" for i in range(len(corpus))]
)
tfidf_df.round(3)

,and,are,cat,cats,dog,dogs,great,log,love,mat,my,on,pet,pets,sat,the
Doc 0,0.000,0.000,0.341,0.000,0.000,0.000,0.000,0.000,0.000,0.432,0.000,0.341,0.000,0.000,0.341,0.682
Doc 1,0.000,0.000,0.000,0.000,0.341,0.000,0.000,0.432,0.000,0.000,0.000,0.341,0.000,0.000,0.341,0.682
Doc 2,0.333,0.422,0.000,0.422,0.000,0.422,0.422,0.000,0.000,0.000,0.000,0.000,0.000,0.422,0.000,0.000
Doc 3,0.239,0.000,0.239,0.000,0.239,0.000,0.000,0.000,0.303,0.000,0.607,0.000,0.607,0.000,0.000,0.000


### IDF scores per word

Higher IDF = rarer / more distinctive word across the corpus.

In [6]:
idf_series = pd.Series(
    tfidf_vectorizer.idf_, index=tfidf_vectorizer.get_feature_names_out()
).sort_values(ascending=False)
idf_series.round(3)

are      1.916
cats     1.916
great    1.916
dogs     1.916
log      1.916
my       1.916
mat      1.916
love     1.916
pets     1.916
pet      1.916
and      1.511
cat      1.511
dog      1.511
on       1.511
sat      1.511
the      1.511
dtype: float64

## 4. Manual TF-IDF (from scratch)

To understand what's happening under the hood, here's a plain-Python reimplementation of TF-IDF — no library black box.

In [7]:
import math
from collections import Counter

# Tokenize
tokenized_docs = [doc.lower().split() for doc in corpus]
vocab = sorted(set(word for doc in tokenized_docs for word in doc))

# Term Frequency (TF) per document
def compute_tf(doc_tokens, vocab):
    counts = Counter(doc_tokens)
    total_terms = len(doc_tokens)
    return {word: counts[word] / total_terms for word in vocab}

tf_scores = [compute_tf(doc, vocab) for doc in tokenized_docs]

# Inverse Document Frequency (IDF) -- standard formula: log(N / df)
def compute_idf(tokenized_docs, vocab):
    N = len(tokenized_docs)
    idf = {}
    for word in vocab:
        df = sum(1 for doc in tokenized_docs if word in doc)
        idf[word] = math.log(N / df) if df else 0.0
    return idf

idf_scores = compute_idf(tokenized_docs, vocab)

# Combine TF * IDF
manual_tfidf = []
for tf in tf_scores:
    manual_tfidf.append({word: round(tf[word] * idf_scores[word], 3) for word in vocab})

manual_tfidf_df = pd.DataFrame(manual_tfidf, index=[f"Doc {i}" for i in range(len(corpus))])
manual_tfidf_df

,and,are,cat,cats,dog,dogs,great,i,log,love,mat,my,on,pet,pets,sat,the
Doc 0,0.000,0.000,0.116,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.231,0.000,0.116,0.000,0.000,0.116,0.231
Doc 1,0.000,0.000,0.000,0.000,0.116,0.000,0.000,0.000,0.231,0.000,0.000,0.000,0.116,0.000,0.000,0.116,0.231
Doc 2,0.116,0.231,0.000,0.231,0.000,0.231,0.231,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.231,0.000,0.000
Doc 3,0.077,0.000,0.077,0.000,0.077,0.000,0.000,0.154,0.000,0.154,0.000,0.308,0.000,0.308,0.000,0.000,0.000


> **Note:** manual values differ slightly from sklearn's because sklearn uses *smoothed* IDF (adds 1 to numerator/denominator) and **L2-normalizes** each document vector by default. Both are valid TF-IDF formulations.

## 5. Summary

**Bag of Words**
- Simple word-count representation.
- Treats every word as equally important.
- Common words (the, on, and) dominate the vector.

**TF-IDF**
- Down-weights words that appear in many documents (less informative).
- Up-weights words that are frequent in a doc but rare across the corpus (more informative / distinctive).
- Generally a stronger feature representation for tasks like search, classification, and clustering.